# Reporte de Performance Logístico — Excel y correo

## Propósito

Generar el reporte ejecutivo de Performance Logístico y Riesgo de Entrega a partir de las vistas KPI de la capa Gold, y enviarlo por correo con el Excel adjunto y un resumen en el cuerpo del mensaje.

## Flujo

1. Registrar el inicio de la ejecución en `etl_log` con estado `EN_PROCESO`.
2. Leer el último corte de cada vista KPI (y el corte anterior del Resumen Ejecutivo para calcular la variación).
3. Construir el Excel de 4 pestañas: RESUMEN EJECUTIVO, DETALLE REGIÓN, DETALLE ORDEN y ALERTAS.
4. Aplicar el formato condicional definido en la especificación del reporte.
5. Generar el gráfico y la tabla resumen del cuerpo del correo.
6. Enviar el correo mediante Microsoft Graph (`sendMail`) con el Excel adjunto.
7. Finalizar el registro en `etl_log` con estado `EXITOSO`.

## Origen

Catálogo `gld_dataco`, esquema `supply_chain`:

* `desc_ft_kpi_resumen_ejecutivo` (último corte y corte anterior)
* `desc_ft_kpi_detalle_region` (último corte)
* `desc_ft_kpi_detalle_orden` (versión vigente)
* `desc_ft_kpi_alertas` (último corte)

## Formato condicional del Excel

| Indicador | Regla |
|---|---|
| On-Time Delivery % | Menor a 70 % en rojo, entre 70 % y 89.9 % en amarillo, desde 90 % en verde |
| Late Delivery Risk % | Mayor a 55 % en rojo |
| Profit Margin % | Menor a 0 (pérdida): fila completa en rojo claro |
| Shipping Variance | Mayor a 5 días: texto rojo en la fila |
| Pedidos sospechosos de fraude | Se incluyen y se marcan en naranja |

## Autenticación

El notebook usa la aplicación de Azure `IngestCloud`. El `SP_TENANT_ID` y el `SP_CLIENT_ID` llegan como parámetros del Job, y el secreto se lee del secret scope `sharepoint` (key `sp_client_secret`). La aplicación necesita el permiso `Mail.Send` de aplicación con consentimiento de administrador.

## Dependencias

Debe ejecutarse como Task del Workflow después de las 4 tasks de KPIs y antes de `finalize_pipeline_control`.

In [0]:
# ============================================================
# GLD_REPORTE_EXCEL_CORREO — Celda 1: Parámetros
# ============================================================

dbutils.widgets.text("gold_catalog", "gld_dataco", "Catálogo Gold")
dbutils.widgets.text("gold_schema", "supply_chain", "Esquema Gold")

# Parámetros del Job (aplicación IngestCloud)
dbutils.widgets.text("SP_TENANT_ID", "")
dbutils.widgets.text("SP_CLIENT_ID", "")

gold_catalog = dbutils.widgets.get("gold_catalog")
gold_schema = dbutils.widgets.get("gold_schema")
SP_TENANT_ID = dbutils.widgets.get("SP_TENANT_ID").strip()
SP_CLIENT_ID = dbutils.widgets.get("SP_CLIENT_ID").strip()

# ----- Correos: se leen del secret scope (no quedan en el código) -----
CORREO_REMITENTE = dbutils.secrets.get(scope="sharepoint", key="correo_remitente").strip()
CORREOS_DESTINATARIOS = [
    c.strip()
    for c in dbutils.secrets.get(scope="sharepoint", key="correo_destinatario").split(",")
    if c.strip()
]

GOLD_KPI_RESUMEN_FQN = f"{gold_catalog}.{gold_schema}.desc_ft_kpi_resumen_ejecutivo"
GOLD_KPI_DETALLE_REGION_FQN = f"{gold_catalog}.{gold_schema}.desc_ft_kpi_detalle_region"
GOLD_KPI_DETALLE_ORDEN_FQN = f"{gold_catalog}.{gold_schema}.desc_ft_kpi_detalle_orden"
GOLD_KPI_ALERTAS_FQN = f"{gold_catalog}.{gold_schema}.desc_ft_kpi_alertas"

RUTA_EXCEL = "/tmp/Performance_Logistico_DataCo.xlsx"

print(f"✓ Remitente configurado: {'sí' if CORREO_REMITENTE else 'NO'}")
print(f"✓ Destinatarios configurados: {len(CORREOS_DESTINATARIOS)}")
print(f"✓ Tenant ID recibido: {'sí' if SP_TENANT_ID else 'no (se leerá del scope)'}")
print(f"✓ Client ID recibido: {'sí' if SP_CLIENT_ID else 'no (se leerá del scope)'}")

In [0]:
# ============================================================
# GLD_REPORTE_EXCEL_CORREO — Celda 2: INICIO DE LOG DE EJECUCIÓN
# ============================================================

import uuid

id_ejecucion = dbutils.jobs.taskValues.get(
    taskKey="prepare_pipeline_control", key="id_ejecucion", default="", debugValue=""
)

LOG_ID = str(uuid.uuid4())
NOTEBOOK = "gld_reporte_excel_correo"
CAPA = "gold"

spark.sql(f"""
INSERT INTO metadata_dataco.supply_chain.hd_etl_log
(log_id, id_ejecucion, notebook, capa, estado, fecha_inicio)
VALUES ('{LOG_ID}', '{id_ejecucion}', '{NOTEBOOK}', '{CAPA}', 'EN_PROCESO', current_timestamp())
""")

dbutils.jobs.taskValues.set(key="log_id", value=LOG_ID)

print(f"✓ Registro EN_PROCESO creado en etl_log: {LOG_ID}")

In [0]:
# ============================================================
# GLD_REPORTE_EXCEL_CORREO — Celda 3: Leer las 4 vistas KPI
# ------------------------------------------------------------
# Resumen Ejecutivo: último corte + corte anterior (variación).
# Detalle Región y Alertas: último corte.
# Detalle Orden: pedidos de la semana actual y anterior.
# ============================================================

import pandas as pd
from pyspark.sql import functions as F

# Corte actual y corte anterior del Resumen Ejecutivo
filas_fechas = (
    spark.table(GOLD_KPI_RESUMEN_FQN)
    .select("fecha_snapshot").distinct()
    .orderBy(F.col("fecha_snapshot").desc())
    .limit(2)
    .collect()
)
fechas_snapshot = [r["fecha_snapshot"] for r in filas_fechas]

if not fechas_snapshot:
    raise ValueError("kpi_resumen_ejecutivo no tiene snapshots. Ejecuta antes las tasks de KPIs.")

fecha_corte = fechas_snapshot[0]
fecha_anterior = fechas_snapshot[1] if len(fechas_snapshot) > 1 else None

df_resumen_sp = spark.table(GOLD_KPI_RESUMEN_FQN)

pdf_resumen = df_resumen_sp.filter(F.col("fecha_snapshot") == F.lit(fecha_corte)).toPandas()

pdf_resumen_ant = (
    df_resumen_sp.filter(F.col("fecha_snapshot") == F.lit(fecha_anterior)).toPandas()
    if fecha_anterior is not None else None
)

pdf_historial = (
    df_resumen_sp.select("fecha_snapshot", "grupo", "on_time_delivery_pct").toPandas()
)

pdf_region = (
    spark.table(GOLD_KPI_DETALLE_REGION_FQN)
    .filter(F.col("es_ultimo_snapshot") == True)
    .orderBy("order_region", "shipping_mode")
    .toPandas()
)

from datetime import timedelta

# Semana actual = 7 días hasta el corte; semana anterior = los 7 días previos
inicio_semana_actual = fecha_corte - timedelta(days=6)
inicio_semana_anterior = fecha_corte - timedelta(days=13)

df_semanas = (
    spark.table("slv_dataco.supply_chain.hd_pedidos")
    .select("order_id", F.to_date("order_date").alias("order_date"))
    .filter(F.col("order_date").between(F.lit(inicio_semana_anterior), F.lit(fecha_corte)))
    .withColumn(
        "semana",
        F.when(F.col("order_date") >= F.lit(inicio_semana_actual), "ACTUAL").otherwise("ANTERIOR")
    )
)

pdf_orden = (
    spark.table(GOLD_KPI_DETALLE_ORDEN_FQN)
    .filter(F.col("es_vigente") == True)
    .join(df_semanas, on="order_id", how="inner")
    .orderBy("semana", "market", "order_region", "order_id")
    .toPandas()
)

pdf_alertas = (
    spark.table(GOLD_KPI_ALERTAS_FQN)
    .filter(F.col("es_ultimo_snapshot") == True)
    .orderBy("market", "order_region", "order_id")
    .toPandas()
)

print(f"✓ Corte actual: {fecha_corte} | Corte anterior: {fecha_anterior}")
print(f"✓ Resumen Ejecutivo: {len(pdf_resumen)} filas")
print(f"✓ Detalle Región: {len(pdf_region)} filas")
print(f"✓ Detalle Orden: {len(pdf_orden)} filas")
print(f"✓ Alertas: {len(pdf_alertas)} filas")

In [0]:
# ============================================================
# GLD_REPORTE_EXCEL_CORREO — Celda 4: Resumen Ejecutivo en formato largo + variación
# ------------------------------------------------------------
# 1 fila por grupo + indicador, con valor actual, valor del corte
# anterior, variación y flecha de tendencia.
# Si no existe corte anterior, la variación queda vacía.
# ============================================================

import pandas as pd

COLOR_VERDE = "2E7D32"
COLOR_ROJO = "C00000"
COLOR_NEUTRO = "7F7F7F"

# columna, etiqueta, tipo de formato, ¿subir es bueno? (None = neutro)
KPIS = [
    ("on_time_delivery_pct", "On-Time Delivery %", "pct", True),
    ("late_delivery_risk_pct", "Late Delivery Risk %", "pct", False),
    ("shipping_variance_avg", "Shipping Variance (días)", "dec", False),
    ("profit_margin_pct", "Profit Margin %", "pctval", True),
    ("revenue_por_cliente", "Revenue por Cliente", "money", True),
    ("beneficio_por_orden", "Beneficio por Orden", "money", True),
    ("lead_time_promedio", "Lead Time Promedio (días)", "dec", False),
    ("total_sospechosos_fraude", "Pedidos sospechosos de fraude", "int", False),
    ("total_pedidos", "Total de pedidos", "int", None),
]

# Los KPIs llegan como double/decimal desde Spark: se fuerzan a numérico
for _col, _, _, _ in KPIS:
    pdf_resumen[_col] = pd.to_numeric(pdf_resumen[_col], errors="coerce")
    if pdf_resumen_ant is not None:
        pdf_resumen_ant[_col] = pd.to_numeric(pdf_resumen_ant[_col], errors="coerce")

ORDEN_GRUPOS = ["GLOBAL", "AMERICAS", "LATAM", "USCA", "EUROPE", "PACIFIC ASIA", "AFRICA"]
grupos_presentes = set(pdf_resumen["grupo"])
grupos = [g for g in ORDEN_GRUPOS if g in grupos_presentes] + sorted(grupos_presentes - set(ORDEN_GRUPOS))

actual = pdf_resumen.set_index("grupo")
anterior = pdf_resumen_ant.set_index("grupo") if pdf_resumen_ant is not None else None


def calcular_fila(grupo, col, tipo, sube_es_bueno):
    """Devuelve valor actual, anterior, variación numérica, flecha y color."""
    act = actual.at[grupo, col]
    act = None if pd.isna(act) else float(act)

    ant = None
    if anterior is not None and grupo in anterior.index:
        v = anterior.at[grupo, col]
        ant = None if pd.isna(v) else float(v)

    if act is None or ant is None:
        return act, ant, None, "—", COLOR_NEUTRO

    delta = act - ant
    if tipo == "pct":
        delta = delta * 100  # puntos porcentuales

    if abs(delta) < 1e-9:
        return act, ant, 0.0, "●", COLOR_NEUTRO

    flecha = "▲" if delta > 0 else "▼"
    if sube_es_bueno is None:
        color = COLOR_NEUTRO
    else:
        color = COLOR_VERDE if (delta > 0) == sube_es_bueno else COLOR_ROJO
    return act, ant, delta, flecha, color


filas_resumen = []
for g in grupos:
    for col, etiqueta, tipo, sube_es_bueno in KPIS:
        act, ant, delta, flecha, color = calcular_fila(g, col, tipo, sube_es_bueno)
        filas_resumen.append({
            "grupo": g, "col": col, "indicador": etiqueta, "tipo": tipo,
            "actual": act, "anterior": ant, "variacion": delta,
            "flecha": flecha, "color": color,
        })

print(f"✓ Resumen en formato largo: {len(filas_resumen)} filas ({len(grupos)} grupos × {len(KPIS)} indicadores)")
print(f"✓ Grupos: {grupos}")
print(f"✓ Variación calculada: {'sí' if anterior is not None else 'no (no existe corte anterior)'}")

In [0]:
%pip install openpyxl

In [0]:
# ============================================================
# GLD_REPORTE_EXCEL_CORREO — Celda 5: Construcción del Excel (4 pestañas)
# ============================================================

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.formatting.rule import FormulaRule
from openpyxl.utils import get_column_letter

# ---------- Estilos ----------
AZUL_OSCURO = "2F5597"   # texto de títulos y grupos
AZUL_PASTEL = "BDD7EE"   # encabezados
AZUL_SUAVE = "DDEBF7"    # celdas de grupo
FILL_ENCABEZADO = PatternFill(start_color=AZUL_PASTEL, end_color=AZUL_PASTEL, fill_type="solid")
FONT_ENCABEZADO = Font(bold=True, color="1F3864")
FONT_TITULO = Font(bold=True, size=14, color=AZUL_OSCURO)
FONT_SUBTITULO = Font(italic=True, size=10, color="595959")
FILL_AZUL_SUAVE = PatternFill(start_color=AZUL_SUAVE, end_color=AZUL_SUAVE, fill_type="solid")

def relleno(hex_color):
    return PatternFill(start_color=hex_color, end_color=hex_color, fill_type="solid")

FILL_ROJO, FONT_ROJO = relleno("FFC7CE"), Font(color="9C0006", bold=True)
FILL_AMARILLO, FONT_AMARILLO = relleno("FFEB9C"), Font(color="9C5700", bold=True)
FILL_VERDE, FONT_VERDE = relleno("C6EFCE"), Font(color="006100", bold=True)
FILL_FILA_PERDIDA = relleno("FFE1E1")   # fila completa con margen negativo
FILL_FRAUDE = relleno("FFD966")         # pedidos sospechosos de fraude
FONT_TEXTO_ROJO = Font(color="C00000")

BORDE = Border(*(Side(style="thin", color="BFBFBF"),) * 4)

FILA_ENCABEZADO = 4
FILA_DATOS = 5

FORMATOS = {
    "pct": '0.0%',            # valor guardado como fracción (0-1)
    "pctval": '0.0"%"',       # valor ya guardado en escala 0-100
    "dec": '0.00',
    "money": '#,##0.00',
    "int": '#,##0',
}
FORMATOS_VARIACION = {
    "pct": '+0.0" pp";-0.0" pp";0.0" pp"',
    "pctval": '+0.0" pp";-0.0" pp";0.0" pp"',
    "dec": '+0.00;-0.00;0.00',
    "money": '+#,##0.00;-#,##0.00;0.00',
    "int": '+#,##0;-#,##0;0',
}

TITULO_REPORTE = "Reporte de Performance Logístico y Riesgo de Entrega"
texto_corte = f"Corte: {fecha_corte}"
if fecha_anterior is not None:
    texto_corte += f"  |  Variación vs corte: {fecha_anterior}"
else:
    texto_corte += "  |  Sin corte anterior: la variación se mostrará desde el siguiente corte"


def preparar_hoja(ws, subtitulo_hoja, encabezados, anchos):
    """Escribe título, subtítulo y encabezado. Los datos se agregan después con ws.append."""
    ws.append([f"{TITULO_REPORTE} — {subtitulo_hoja}"])
    ws.append([texto_corte])
    ws.append([])
    ws.append(encabezados)
    ws["A1"].font = FONT_TITULO
    ws["A2"].font = FONT_SUBTITULO
    for c in range(1, len(encabezados) + 1):
        celda = ws.cell(row=FILA_ENCABEZADO, column=c)
        celda.fill = FILL_ENCABEZADO
        celda.font = FONT_ENCABEZADO
        celda.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        ws.column_dimensions[get_column_letter(c)].width = anchos[c - 1]
    ws.row_dimensions[FILA_ENCABEZADO].height = 32
    ws.freeze_panes = ws.cell(row=FILA_DATOS, column=1)
    ws.sheet_view.showGridLines = False


def filas_para_excel(df):
    """DataFrame a lista de filas, con NaN/NaT como None."""
    return df.astype(object).where(pd.notna(df), None).values.tolist()


def si_no(serie):
    return serie.map(lambda v: "SÍ" if v is True or v == 1 else ("NO" if v is False or v == 0 else ""))


def regla(ws, rango, formula, fill=None, font=None):
    ws.conditional_formatting.add(rango, FormulaRule(formula=[formula], fill=fill, font=font))


wb = Workbook()

# ====================================================================
# Pestaña 1: RESUMEN EJECUTIVO
# ====================================================================
ws1 = wb.active
ws1.title = "RESUMEN EJECUTIVO"
preparar_hoja(
    ws1, "RESUMEN EJECUTIVO",
    ["Mercado", "Indicador", "Valor actual", "Valor anterior", "Variación", "Tendencia"],
    [18, 34, 16, 16, 16, 12],
)

fila = FILA_DATOS
inicio_bloque = FILA_DATOS
grupo_previo = None

for f in filas_resumen:
    if grupo_previo is not None and f["grupo"] != grupo_previo:
        if fila - 1 > inicio_bloque:
            ws1.merge_cells(start_row=inicio_bloque, start_column=1, end_row=fila - 1, end_column=1)
        inicio_bloque = fila
    grupo_previo = f["grupo"]

    ws1.cell(row=fila, column=1, value=f["grupo"])
    ws1.cell(row=fila, column=2, value=f["indicador"])
    c_act = ws1.cell(row=fila, column=3, value=f["actual"])
    c_ant = ws1.cell(row=fila, column=4, value=f["anterior"])
    c_var = ws1.cell(row=fila, column=5, value=f["variacion"])
    c_fle = ws1.cell(row=fila, column=6, value=f["flecha"])

    c_act.number_format = FORMATOS[f["tipo"]]
    c_ant.number_format = FORMATOS[f["tipo"]]
    c_var.number_format = FORMATOS_VARIACION[f["tipo"]]
    c_fle.font = Font(bold=True, color=f["color"])
    c_fle.alignment = Alignment(horizontal="center")

    # ----- Formato condicional del requerimiento -----
    valor = f["actual"]
    if valor is not None:
        if f["col"] == "on_time_delivery_pct":
            if valor < 0.70:
                c_act.fill, c_act.font = FILL_ROJO, FONT_ROJO
            elif valor < 0.90:
                c_act.fill, c_act.font = FILL_AMARILLO, FONT_AMARILLO
            else:
                c_act.fill, c_act.font = FILL_VERDE, FONT_VERDE
        elif f["col"] == "late_delivery_risk_pct" and valor > 0.55:
            c_act.fill, c_act.font = FILL_ROJO, FONT_ROJO
        elif f["col"] == "profit_margin_pct" and valor < 0:
            for c in range(2, 7):
                ws1.cell(row=fila, column=c).fill = FILL_FILA_PERDIDA
        elif f["col"] == "shipping_variance_avg" and valor > 5:
            for c in (2, 3, 4):
                ws1.cell(row=fila, column=c).font = FONT_TEXTO_ROJO

    for c in range(1, 7):
        ws1.cell(row=fila, column=c).border = BORDE
    fila += 1

if fila - 1 > inicio_bloque:
    ws1.merge_cells(start_row=inicio_bloque, start_column=1, end_row=fila - 1, end_column=1)

for r in range(FILA_DATOS, fila):
    celda_grupo = ws1.cell(row=r, column=1)
    celda_grupo.font = Font(bold=True, color=AZUL_OSCURO)
    celda_grupo.alignment = Alignment(horizontal="center", vertical="center")

# ====================================================================
# Pestaña 2: DETALLE REGIÓN
# ====================================================================
cols_region = [
    "order_region", "shipping_mode", "on_time_delivery_pct", "late_delivery_risk_pct",
    "shipping_variance_avg", "profit_margin_pct", "revenue_por_cliente",
    "beneficio_por_orden", "lead_time_promedio", "total_sospechosos_fraude", "total_pedidos",
]
for c in cols_region[2:]:
    pdf_region[c] = pd.to_numeric(pdf_region[c], errors="coerce")
df_r = pdf_region[cols_region]

ws2 = wb.create_sheet("DETALLE REGIÓN")
preparar_hoja(
    ws2, "DETALLE REGIÓN",
    ["Región", "Modo de envío", "On-Time Delivery %", "Late Delivery Risk %",
     "Shipping Variance (días)", "Profit Margin %", "Revenue por Cliente",
     "Beneficio por Orden", "Lead Time Promedio (días)", "Pedidos sospechosos de fraude",
     "Total de pedidos"],
    [26, 18, 16, 16, 16, 14, 16, 16, 16, 18, 14],
)
for fila_datos in filas_para_excel(df_r):
    ws2.append(fila_datos)

ultima_r = max(FILA_DATOS, FILA_ENCABEZADO + len(df_r))
formatos_r = {3: "pct", 4: "pct", 5: "dec", 6: "pctval", 7: "money", 8: "money", 9: "dec", 10: "int", 11: "int"}
for r in range(FILA_DATOS, ultima_r + 1):
    for col_idx, tipo in formatos_r.items():
        ws2.cell(row=r, column=col_idx).number_format = FORMATOS[tipo]
    for c in range(1, 12):
        ws2.cell(row=r, column=c).border = BORDE

# Merge de la región (columna A) cuando se repite
r = FILA_DATOS
while r <= ultima_r and len(df_r) > 0:
    valor = ws2.cell(row=r, column=1).value
    fin = r
    while fin + 1 <= ultima_r and ws2.cell(row=fin + 1, column=1).value == valor:
        fin += 1
    if fin > r:
        ws2.merge_cells(start_row=r, start_column=1, end_row=fin, end_column=1)
    ws2.cell(row=r, column=1).alignment = Alignment(vertical="center")
    ws2.cell(row=r, column=1).font = Font(bold=True, color=AZUL_OSCURO)
    r = fin + 1

rango_r = f"A{FILA_DATOS}:K{ultima_r}"
# Orden de prioridad: primero las reglas de celda, después las de fila
regla(ws2, f"C{FILA_DATOS}:C{ultima_r}", f"AND(ISNUMBER($C{FILA_DATOS}),$C{FILA_DATOS}<0.7)", FILL_ROJO, FONT_ROJO)
regla(ws2, f"C{FILA_DATOS}:C{ultima_r}", f"AND(ISNUMBER($C{FILA_DATOS}),$C{FILA_DATOS}>=0.7,$C{FILA_DATOS}<0.9)", FILL_AMARILLO, FONT_AMARILLO)
regla(ws2, f"C{FILA_DATOS}:C{ultima_r}", f"AND(ISNUMBER($C{FILA_DATOS}),$C{FILA_DATOS}>=0.9)", FILL_VERDE, FONT_VERDE)
regla(ws2, f"D{FILA_DATOS}:D{ultima_r}", f"AND(ISNUMBER($D{FILA_DATOS}),$D{FILA_DATOS}>0.55)", FILL_ROJO, FONT_ROJO)
regla(ws2, rango_r, f"AND(ISNUMBER($F{FILA_DATOS}),$F{FILA_DATOS}<0)", FILL_FILA_PERDIDA, None)
regla(ws2, rango_r, f"AND(ISNUMBER($E{FILA_DATOS}),$E{FILA_DATOS}>5)", None, FONT_TEXTO_ROJO)
ws2.auto_filter.ref = f"A{FILA_ENCABEZADO}:K{ultima_r}"

# ====================================================================
# Pestaña 3: DETALLE ORDEN
# ====================================================================
cols_orden = [
    "order_id", "market", "order_region", "customer_segment", "shipping_mode",
    "order_status", "delivery_status", "total_sales", "total_profit",
    "profit_margin_pct", "avg_benefit_per_order", "total_quantity",
    "es_entrega_a_tiempo", "shipping_variance", "es_sospechoso_fraude",
    "fecha_inicio_vigencia", "semana", "order_date",
]
df_o = pdf_orden[cols_orden].copy()
for c in ["total_sales", "total_profit", "profit_margin_pct", "avg_benefit_per_order",
          "total_quantity", "shipping_variance"]:
    df_o[c] = pd.to_numeric(df_o[c], errors="coerce")
df_o["es_entrega_a_tiempo"] = si_no(df_o["es_entrega_a_tiempo"])
df_o["es_sospechoso_fraude"] = si_no(df_o["es_sospechoso_fraude"])
df_o["fecha_inicio_vigencia"] = df_o["fecha_inicio_vigencia"].astype(str)
df_o["order_date"] = df_o["order_date"].astype(str)

ws3 = wb.create_sheet("DETALLE ORDEN")
preparar_hoja(
    ws3, "DETALLE ORDEN",
    ["Pedido", "Mercado", "Región", "Segmento de cliente", "Modo de envío",
     "Estado del pedido", "Estado de entrega", "Ventas", "Utilidad", "Margen %",
     "Beneficio por orden", "Cantidad", "Entrega a tiempo", "Shipping Variance (días)",
      "Sospechoso de fraude", "Vigente desde", "Semana", "Fecha del pedido"],
    [12, 16, 24, 18, 16, 18, 20, 12, 12, 11, 14, 10, 12, 14, 14, 14, 12, 16],
)
ws3["A2"] = (
    f"{texto_corte}  |  Pedidos de la semana ACTUAL ({inicio_semana_actual} → {fecha_corte}) "
    f"y ANTERIOR ({inicio_semana_anterior} → {inicio_semana_actual - timedelta(days=1)})"
)
for fila_datos in filas_para_excel(df_o):
    ws3.append(fila_datos)

ultima_o = max(FILA_DATOS, FILA_ENCABEZADO + len(df_o))
formatos_o = {8: "money", 9: "money", 10: "pctval", 11: "money", 12: "int", 14: "dec"}
for r in range(FILA_DATOS, ultima_o + 1):
    for col_idx, tipo in formatos_o.items():
        ws3.cell(row=r, column=col_idx).number_format = FORMATOS[tipo]

rango_o = f"A{FILA_DATOS}:R{ultima_o}"
regla(ws3, f"A{FILA_DATOS}:A{ultima_o}", f'$O{FILA_DATOS}="SÍ"', FILL_FRAUDE, Font(bold=True))
regla(ws3, rango_o, f"AND(ISNUMBER($J{FILA_DATOS}),$J{FILA_DATOS}<0)", FILL_FILA_PERDIDA, None)
regla(ws3, rango_o, f"AND(ISNUMBER($N{FILA_DATOS}),$N{FILA_DATOS}>5)", None, FONT_TEXTO_ROJO)
ws3.auto_filter.ref = f"A{FILA_ENCABEZADO}:R{ultima_o}"

# ====================================================================
# Pestaña 4: ALERTAS
# ====================================================================
cols_alertas = [
    "order_id", "market", "order_region", "customer_segment", "order_status",
    "delivery_status", "late_delivery_risk_original", "late_delivery_risk_calculado",
    "riesgo_validado", "es_sospechoso_fraude", "motivo_alerta",
    "total_sales", "total_profit", "fecha_fin_corte",
]
df_a = pdf_alertas[cols_alertas].copy()
for c in ["total_sales", "total_profit"]:
    df_a[c] = pd.to_numeric(df_a[c], errors="coerce")
df_a["riesgo_validado"] = si_no(df_a["riesgo_validado"])
df_a["es_sospechoso_fraude"] = si_no(df_a["es_sospechoso_fraude"])
df_a["fecha_fin_corte"] = df_a["fecha_fin_corte"].astype(str)

ws4 = wb.create_sheet("ALERTAS")
preparar_hoja(
    ws4, "ALERTAS",
    ["Pedido", "Mercado", "Región", "Segmento de cliente", "Estado del pedido",
     "Estado de entrega", "Riesgo de entrega (origen)", "Riesgo de entrega (calculado)",
     "Riesgo validado", "Sospechoso de fraude", "Motivo de la alerta",
     "Ventas", "Utilidad", "Fecha de corte"],
    [12, 16, 24, 18, 18, 20, 14, 14, 12, 14, 38, 12, 12, 14],
)
for fila_datos in filas_para_excel(df_a):
    ws4.append(fila_datos)

ultima_a = max(FILA_DATOS, FILA_ENCABEZADO + len(df_a))
for r in range(FILA_DATOS, ultima_a + 1):
    ws4.cell(row=r, column=12).number_format = FORMATOS["money"]
    ws4.cell(row=r, column=13).number_format = FORMATOS["money"]

rango_a = f"A{FILA_DATOS}:N{ultima_a}"
regla(ws4, f"A{FILA_DATOS}:A{ultima_a}", f'$J{FILA_DATOS}="SÍ"', FILL_FRAUDE, Font(bold=True))
regla(ws4, rango_a, f"AND(ISNUMBER($M{FILA_DATOS}),$M{FILA_DATOS}<0)", FILL_FILA_PERDIDA, None)
ws4.auto_filter.ref = f"A{FILA_ENCABEZADO}:N{ultima_a}"

# ---------- Guardar ----------
wb.save(RUTA_EXCEL)

import os
tamano_bytes = os.path.getsize(RUTA_EXCEL)
print(f"✓ Excel generado: {RUTA_EXCEL}")
print(f"✓ Tamaño: {tamano_bytes / 1024 / 1024:.2f} MB")
print(f"✓ Filas — Resumen: {len(filas_resumen)} | Región: {len(df_r)} | Orden: {len(df_o)} | Alertas: {len(df_a)}")

# Microsoft Graph acepta adjuntos de hasta 3 MB en una sola llamada a sendMail
if tamano_bytes > 3_000_000:
    raise ValueError(
        f"El Excel pesa {tamano_bytes / 1024 / 1024:.2f} MB y supera el límite de 3 MB "
        "para adjuntos en sendMail. Reduce el detalle de la pestaña DETALLE ORDEN."
    )

In [0]:
# ============================================================
# GLD_REPORTE_EXCEL_CORREO — Celda 6: Gráfico del cuerpo del correo
# ------------------------------------------------------------
# Con 2 o más cortes: tendencia de On-Time Delivery % por grupo.
# Con 1 solo corte: barras de On-Time Delivery % por grupo.
# ============================================================

import io
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


def generar_grafico_otd():
    hist = pdf_historial.copy()
    hist["on_time_delivery_pct"] = pd.to_numeric(hist["on_time_delivery_pct"], errors="coerce") * 100
    fechas = sorted(hist["fecha_snapshot"].unique())

    fig, ax = plt.subplots(figsize=(8, 3.8), dpi=110)

    if len(fechas) >= 2:
        for g in grupos:
            d = hist[hist["grupo"] == g].sort_values("fecha_snapshot")
            ax.plot(
                [str(x) for x in d["fecha_snapshot"]], d["on_time_delivery_pct"],
                marker="o", linewidth=2.8 if g == "GLOBAL" else 1.2, label=g,
            )
        ax.axhline(70, color="#C00000", linestyle="--", linewidth=0.9)
        ax.axhline(90, color="#2E7D32", linestyle="--", linewidth=0.9)
        ax.set_ylabel("On-Time Delivery %")
        ax.set_title("Tendencia de On-Time Delivery % por corte")
        ax.legend(fontsize=7, ncol=4, loc="lower center", bbox_to_anchor=(0.5, -0.32), frameon=False)
    else:
        d = (
            hist[hist["fecha_snapshot"] == fechas[0]]
            .set_index("grupo").reindex(grupos)["on_time_delivery_pct"]
        )
        colores = [
            "#C00000" if v < 70 else "#E6A700" if v < 90 else "#2E7D32"
            for v in d.fillna(0).values
        ]
        ax.barh(list(d.index)[::-1], list(d.values)[::-1], color=colores[::-1])
        ax.axvline(70, color="#C00000", linestyle="--", linewidth=0.9)
        ax.axvline(90, color="#2E7D32", linestyle="--", linewidth=0.9)
        ax.set_xlim(0, 100)
        ax.set_xlabel("On-Time Delivery %")
        ax.set_title(f"On-Time Delivery % por mercado — corte {fechas[0]}")
        for i, v in enumerate(list(d.values)[::-1]):
            if pd.notna(v):
                ax.text(min(v + 1, 92), i, f"{v:.1f}%", va="center", fontsize=8)

    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", bbox_inches="tight")
    plt.close(fig)
    return buffer.getvalue()


grafico_png = generar_grafico_otd()
print(f"✓ Gráfico generado ({len(grafico_png) / 1024:.0f} KB)")

In [0]:
# ============================================================
# GLD_REPORTE_EXCEL_CORREO — Celda 7: Cuerpo HTML del correo
# ------------------------------------------------------------
# Tabla resumen (7 KPIs + pedidos por mercado) con semáforos
# y variación. El gráfico va como imagen incrustada (cid).
# ============================================================

def fmt_valor(valor, tipo):
    if valor is None or pd.isna(valor):
        return "—"
    if tipo == "pct":
        return f"{valor * 100:.1f}%"
    if tipo == "pctval":
        return f"{valor:.1f}%"
    if tipo == "dec":
        return f"{valor:.2f}"
    if tipo == "money":
        return f"{valor:,.2f}"
    return f"{int(round(valor)):,}"


def fmt_variacion(f):
    if f["variacion"] is None:
        return ""
    if f["tipo"] in ("pct", "pctval"):
        texto = f"{f['variacion']:+.1f} pp"
    elif f["tipo"] == "int":
        texto = f"{f['variacion']:+,.0f}"
    else:
        texto = f"{f['variacion']:+,.2f}"
    return (
        f'<br><span style="font-size:10px;color:#{f["color"]};">'
        f'{f["flecha"]} {texto}</span>'
    )


def estilo_semaforo(col, valor):
    """Colores de fondo/texto de la celda según el formato condicional del reporte."""
    if valor is None:
        return ""
    if col == "on_time_delivery_pct":
        if valor < 0.70:
            return "background:#FFC7CE;color:#9C0006;font-weight:bold;"
        if valor < 0.90:
            return "background:#FFEB9C;color:#9C5700;font-weight:bold;"
        return "background:#C6EFCE;color:#006100;font-weight:bold;"
    if col == "late_delivery_risk_pct" and valor > 0.55:
        return "background:#FFC7CE;color:#9C0006;font-weight:bold;"
    if col == "profit_margin_pct" and valor < 0:
        return "background:#FFE1E1;"
    if col == "shipping_variance_avg" and valor > 5:
        return "color:#C00000;"
    return ""


indice = {(f["grupo"], f["col"]): f for f in filas_resumen}

TH = "padding:6px 8px;background:#1F3864;color:#FFFFFF;font-size:11px;text-align:center;"
TD = "padding:6px 8px;border-bottom:1px solid #D9D9D9;font-size:12px;text-align:center;"

encabezados_html = "".join(f'<th style="{TH}">{etiqueta}</th>' for _, etiqueta, _, _ in KPIS)

filas_html = ""
for g in grupos:
    celdas = ""
    for col, _, tipo, _ in KPIS:
        f = indice[(g, col)]
        celdas += (
            f'<td style="{TD}{estilo_semaforo(col, f["actual"])}">'
            f'{fmt_valor(f["actual"], tipo)}{fmt_variacion(f)}</td>'
        )
    filas_html += (
        f'<tr><td style="{TD}text-align:left;font-weight:bold;color:#1F3864;">{g}</td>{celdas}</tr>'
    )

tabla_html = (
    '<table style="border-collapse:collapse;font-family:Segoe UI,Arial,sans-serif;">'
    f'<tr><th style="{TH}">Mercado</th>{encabezados_html}</tr>{filas_html}</table>'
)

global_otd = indice[("GLOBAL", "on_time_delivery_pct")]["actual"] if ("GLOBAL", "on_time_delivery_pct") in indice else None
global_pedidos = indice[("GLOBAL", "total_pedidos")]["actual"] if ("GLOBAL", "total_pedidos") in indice else None
n_alertas = len(df_a)

resumen_texto = f"Corte al <b>{fecha_corte}</b>."
if global_pedidos is not None:
    resumen_texto += f" Pedidos analizados: <b>{fmt_valor(global_pedidos, 'int')}</b>."
if global_otd is not None:
    resumen_texto += f" On-Time Delivery global: <b>{fmt_valor(global_otd, 'pct')}</b>."
resumen_texto += f" Pedidos en alerta: <b>{n_alertas:,}</b>."

nota_variacion = (
    ""
    if fecha_anterior is not None
    else '<p style="font-size:11px;color:#7F7F7F;">Es el primer corte del historial, por eso aún no se muestra variación.</p>'
)

cuerpo_html = f"""
<html>
<body style="font-family:Segoe UI,Arial,sans-serif;color:#262626;">
  <h2 style="color:#1F3864;margin-bottom:4px;">Performance Logístico y Riesgo de Entrega</h2>
  <p style="margin-top:0;">{resumen_texto}</p>
  {tabla_html}
  {nota_variacion}
  <br>
  <img src="cid:grafico_otd" alt="On-Time Delivery %" style="max-width:640px;">
  <p style="font-size:12px;">El Excel completo (4 pestañas: Resumen Ejecutivo, Detalle Región, Detalle Orden y Alertas) se adjunta a este correo.</p>
  <p style="font-size:11px;color:#7F7F7F;">Generado automáticamente por el pipeline DataCo Lakehouse.</p>
</body>
</html>
"""

print("✓ Cuerpo HTML generado")

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
w.secrets.put_secret(scope="sharepoint", key="sp_tenant_id", string_value="dbbe4c70-5af0-4277-8e8b-261701ee1145")
w.secrets.put_secret(scope="sharepoint", key="sp_client_id", string_value="dfac7cc6-c030-4bc9-a580-2ac6f813d3f9")

print("✓ Ids guardados en el scope sharepoint")

In [0]:
# ============================================================
# GLD_REPORTE_EXCEL_CORREO — Celda 8: Envío por Microsoft Graph
# ============================================================

print("Celda 8 — versión con lectura desde el scope")

import requests
import base64

# Prioridad: parámetro del Job (widget). Si está vacío, se lee del secret scope.
def leer_parametro(widget, clave_secret):
    try:
        valor = dbutils.widgets.get(widget).strip()
    except Exception:
        valor = ""
    if valor:
        return valor
    try:
        return dbutils.secrets.get(scope="sharepoint", key=clave_secret).strip()
    except Exception:
        return ""

SP_TENANT_ID = leer_parametro("SP_TENANT_ID", "sp_tenant_id")
SP_CLIENT_ID = leer_parametro("SP_CLIENT_ID", "sp_client_id")

if not SP_TENANT_ID or not SP_CLIENT_ID:
    raise ValueError(
        "Faltan SP_TENANT_ID o SP_CLIENT_ID: los cuadros del notebook están vacíos "
        "y no existen sp_tenant_id / sp_client_id en el scope sharepoint."
    )

# El secreto se lee directo del secret scope
SP_CLIENT_SECRET = dbutils.secrets.get(scope="sharepoint", key="sp_client_secret")

# ----- Token (client credentials) -----
token_response = requests.post(
    f"https://login.microsoftonline.com/{SP_TENANT_ID}/oauth2/v2.0/token",
    data={
        "grant_type": "client_credentials",
        "client_id": SP_CLIENT_ID,
        "client_secret": SP_CLIENT_SECRET,
        "scope": "https://graph.microsoft.com/.default",
    },
    timeout=30,
)

if token_response.status_code != 200:
    raise RuntimeError(
        f"Error al obtener token de Microsoft Graph: "
        f"{token_response.status_code} - {token_response.text}"
    )

access_token = token_response.json()["access_token"]
print("✓ Token de Microsoft Graph obtenido")

# ----- Adjuntos -----
with open(RUTA_EXCEL, "rb") as f:
    excel_base64 = base64.b64encode(f.read()).decode("utf-8")

grafico_base64 = base64.b64encode(grafico_png).decode("utf-8")

mensaje = {
    "message": {
        "subject": f"Performance Logístico DataCo — Corte al {fecha_corte}",
        "body": {"contentType": "HTML", "content": cuerpo_html},
        "toRecipients": [{"emailAddress": {"address": c}} for c in CORREOS_DESTINATARIOS],
        "attachments": [
            {
                "@odata.type": "#microsoft.graph.fileAttachment",
                "name": f"Performance_Logistico_DataCo_{fecha_corte}.xlsx",
                "contentType": "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
                "contentBytes": excel_base64,
            },
            {
                "@odata.type": "#microsoft.graph.fileAttachment",
                "name": "grafico_otd.png",
                "contentType": "image/png",
                "contentBytes": grafico_base64,
                "isInline": True,
                "contentId": "grafico_otd",
            },
        ],
    },
    "saveToSentItems": True,
}

send_response = requests.post(
    f"https://graph.microsoft.com/v1.0/users/{CORREO_REMITENTE}/sendMail",
    headers={"Authorization": f"Bearer {access_token}", "Content-Type": "application/json"},
    json=mensaje,
    timeout=60,
)

if send_response.status_code != 202:
    raise RuntimeError(
        f"Error al enviar el correo ({send_response.status_code}): {send_response.text}"
    )

print(f"✓ Correo enviado a: {', '.join(CORREOS_DESTINATARIOS)}")

In [0]:
spark.sql(f"""
UPDATE metadata_dataco.supply_chain.hd_etl_log
SET estado = 'EXITOSO',
    fecha_fin = current_timestamp(),
    filas_procesadas = {len(df_o)}
WHERE log_id = '{LOG_ID}'
""")

print("✓ etl_log actualizado: EXITOSO")
print(f"✓ Filas procesadas (detalle de órdenes): {len(df_o)}")